In [ ]:
# ---------------------------------------------------------
# 1. INSTALLS & IMPORTS
# ---------------------------------------------------------
!pip install -q --upgrade pip
!pip install -q openai-whisper yt-dlp ffmpeg-python pytesseract opencv-python sentence-transformers google-generativeai gradio
!pip install nest_asyncio
import os
import torch
import whisper
import cv2
import pytesseract
import numpy as np
import gradio as gr
import subprocess
import nest_asyncio
import pandas as pd
import re
import time
from skimage.metrics import structural_similarity as ssim
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai

nest_asyncio.apply()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ---------------------------------------------------------
# 1. CONFIG & MODELS (Multi-Modal Intelligence)
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = whisper.load_model("base").to(device)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# SECURITY: Replace with your actual API key securely
GEMINI_API_KEY = ""
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash") # Updated to stable version string

# ---------------------------------------------------------
# 2. CORE PIPELINE (Adaptive & Forensic)
# ---------------------------------------------------------

def process_video_streaming(video_file):
    logs = []
    def update_logs(msg):
        logs.append(msg)
        return "\n".join(logs)

    if not video_file:
        yield update_logs("❌ No file uploaded."), None, None
        return

    try:
        start_time = time.perf_counter()
        # --- AUDIO EXTRACTION ---
        yield update_logs("🎙️ Step 1: Extracting Audio via FFmpeg..."), None, None
        os.makedirs("data", exist_ok=True)
        audio_path = "data/audio.wav"
        subprocess.call(f'ffmpeg -i "{video_file.name}" -ab 192k -ac 2 -ar 44100 -vn -y {audio_path}', shell=True)

        # --- WHISPER TRANSCRIPTION ---
        yield update_logs("🗣️ Step 2: Transcribing Speech (Whisper ASR)..."), None, None
        asr_res = asr_model.transcribe(audio_path)
        yield update_logs(f"✅ Transcribed {len(asr_res['segments'])} audio segments."), None, None

        # --- ADAPTIVE SCENE DETECTION (Z-Score) ---
        yield update_logs("🔍 Step 3: Statistical Visual Profiling..."), None, None
        cap = cv2.VideoCapture(video_file.name)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_seconds = total_f / fps

        # We sample the video to find the "normal" rate of visual change
        step = 15
        sample_limit = max(50, int(total_f * 0.05 / step))
        ssim_scores, prev_gray = [], None
        for _ in range(sample_limit):
            ret, frame = cap.read()
            if not ret: break
            gray = cv2.cvtColor(cv2.resize(frame, (480, 270)), cv2.COLOR_BGR2GRAY)
            if prev_gray is not None: ssim_scores.append(ssim(prev_gray, gray))
            prev_gray = gray
            for _ in range(step - 1): cap.grab()

        mu, sigma = np.mean(ssim_scores), np.std(ssim_scores)
        threshold = mu - (2 * sigma) # Detect changes 2 standard deviations from mean

        # --- KEYFRAME & OCR EXTRACTION ---
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        keyframes, idx, prev_gray = [], 0, None
        while True:
            ret, frame = cap.read()
            if not ret: break
            if idx % step == 0:
                gray = cv2.cvtColor(cv2.resize(frame, (480, 270)), cv2.COLOR_BGR2GRAY)
                if prev_gray is not None and ssim(prev_gray, gray) < threshold:
                    keyframes.append((idx / fps, frame))
                prev_gray = gray
            idx += 1
        cap.release()

        yield update_logs(f"🎬 Found {len(keyframes)} logical scene transitions."), None, None
        ocr_results, confidences = [], []
        for ts, frame in keyframes:
            data = pytesseract.image_to_data(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), output_type=pytesseract.Output.DICT)
            words = [data['text'][i] for i, c in enumerate(data['conf']) if int(c) > 75]
            if words: confidences.append(np.mean([int(c) for c in data['conf'] if int(c) > 0]))
            txt = " ".join(words).strip()
            if len(txt) > 15: ocr_results.append((ts, txt))

        # --- SEMANTIC CHUNKING ---
        yield update_logs("🧩 Step 5: Semantic Topic Partitioning..."), None, None
        combined = [(s['start'], s['text']) for s in asr_res['segments']] + ocr_results
        combined.sort(key=lambda x: x[0]) # Synchronize Sight and Sound chronologically

        texts = [x[1] for x in combined]
        embs = embedder.encode(texts)
        adj_sims = [cosine_similarity([embs[i]], [embs[i+1]])[0][0] for i in range(len(embs)-1)]
        shift_t = np.percentile(adj_sims, 15) # Find the mathematical "elbow" of topic changes

        chunks = []
        cur_c = []  # will store (timestamp, text)

        for i in range(len(adj_sims)):
          # Add the current (timestamp, text)
          cur_c.append(combined[i])

          # Check for topic shift AND minimum chunk size
          if adj_sims[i] < shift_t and len(" ".join(t for _, t in cur_c)) > 250:
            # ---- COMPUTE REPRESENTATIVE TIMESTAMP (MEDIAN) ----
            times = [t for t, _ in cur_c]
            rep_time = float(np.median(times))

            # ---- MERGE TEXT ----
            chunk_text = " ".join(t for _, t in cur_c)

            # ---- SAVE CHUNK ----
            chunks.append((rep_time, chunk_text))

            # ---- RESET ----
            cur_c = []

        # Handle final chunk
        if cur_c:
          times = [t for t, _ in cur_c]
          rep_time = float(np.median(times))
          chunk_text = " ".join(t for _, t in cur_c)
          chunks.append((rep_time, chunk_text))

        # --- DYNAMIC MMR SYNTHESIS (NO HARDCODING) ---
        yield update_logs("⚖️ Step 6: Fluid Information Diversity Optimization..."), None, None

        # Calculate dynamic limit: 1 chunk per 2 minutes of video + 20% of total semantic variety
        duration_minutes = duration_seconds / 60
        time_based_limit = max(4, int(duration_minutes / 2))
        content_based_limit = int(len(chunks) * 0.25)
        dynamic_limit = min(40, max(time_based_limit, content_based_limit))

        q_emb = embedder.encode(["Key definitions, core technical concepts, important formulas, algorithms, and concise lecture summaries"])
        c_texts = [c[1] for c in chunks]
        c_embs = embedder.encode(c_texts)
        relevance_scores = cosine_similarity(q_emb, c_embs)[0]

        selected_indices = []
        remaining_indices = list(range(len(chunks)))
        lambda_param = 0.5 # Balance Diversity vs Relevance

        while len(selected_indices) < dynamic_limit and remaining_indices:
            mmr_scores = []
            for idx in remaining_indices:
                rel = relevance_scores[idx]
                red = max([cosine_similarity([c_embs[idx]], [c_embs[s]])[0][0] for s in selected_indices]) if selected_indices else 0
                score = lambda_param * rel - (1 - lambda_param) * red
                mmr_scores.append((score, idx))

            best_idx = max(mmr_scores, key=lambda x: x[0])[1]
            selected_indices.append(best_idx)
            remaining_indices.remove(best_idx)

        # Sort indices chronologically before passing to Gemini
        selected_indices.sort()
        context = "\n\n".join([f"[{chunks[i][0]:.2f}s]: {chunks[i][1]}" for i in selected_indices])



        # --- FINAL FORMATTED GENERATION ---
        yield update_logs(f"🧠 Step 7: Final Synthesis (Captured {len(selected_indices)} key segments)..."), None, None

        prompt = f"""
        Extract the knowledge from this lecture context.
        Structure your response EXACTLY like this:

        ### Summary: [Creative Title Based on Context]
        [Detailed 3-paragraph summary. Use **bolding** for key terms.
        IMPORTANT: You MUST cite the source time for every claim using [0.00s] format.]

        ---
        ### Main Thesis of the Lecture
        [One sentence bolded summary of the core message]

        ---
        ### Key Technical Takeaways
        * **[Concept Name]**: [Explanation] [Timestamp]
        * [Add more bullet points based on the richness of the context]

        CONTEXT:
        {context}
        """

        print(prompt)

        response = gemini_model.generate_content(prompt).text
        end_time = time.perf_counter()
        total_time = end_time - start_time


        # --- EVALUATION METRICS ---
        citations = len(re.findall(r'\[\d+\.\d+s\]', response))
        grounding_score = min(100, (citations / (dynamic_limit * 0.8)) * 100)

        vitals_data = [
            ["Visual Volatility", f"{len(keyframes)/duration_seconds:.2f}", "Slide changes per second."],
            ["Temporal Depth", f"{duration_minutes:.1f} min", "Total length analyzed."],
            ["Semantic Chapters", f"{len(chunks)}", "Total unique topics discovered."]
        ]

        eval_data = [
            ["OCR Legibility", f"{np.mean(confidences):.1f}%" if confidences else "N/A", "Slide text clarity."],
            ["Source Grounding", f"{grounding_score:.1f}/100", "Evidence-based truthfulness score."],
            ["Context Utilization", f"{(len(selected_indices)/len(chunks))*100:.1f}%", "Percentage of lecture topics synthesized."]
        ]

        yield update_logs("✅ Analysis Complete."), pd.DataFrame(vitals_data, columns=["Metric", "Value", "Insights"]), pd.DataFrame(eval_data, columns=["Eval Metric", "Result", "Explanation"])

        logs.append("\n" + "="*40 + "\n" + response)
        yield update_logs(f"⏱️ Total End-to-End Processing Time: {total_time:.2f} seconds"), None, None
        yield "\n".join(logs), pd.DataFrame(vitals_data, columns=["Metric", "Value", "Insights"]), pd.DataFrame(eval_data, columns=["Eval Metric", "Result", "Explanation"])

    except Exception as e:
        yield update_logs(f"❌ Pipeline Failure: {str(e)}"), None, None

# ---------------------------------------------------------
# 3. GRADIO UI
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"), title="LectaLens") as demo:
    gr.Markdown("# 🎓 LectaLens: Because great lectures are meant to be seen, not just heard.")
    gr.Markdown("What makes me different and intelligent? ")
    gr.Markdown("""
    Most tools are **'Blind'**—they only listen to the audio. **LectaLens AI** is a **Multi-Modal Lecture Intelligence System* that synchronizes sight and sound.

    | Feature | Traditional Tools | **LectaLens AI** |
    | :--- | :--- | :--- |
    | **Sensory Input** | Ears Only (Audio) | **Ears + Eyes (Audio & Visuals)** |
    | **Topic Logic** | Word Count Cuts | **Semantic Quantile Partitioning** |
    | **Context** | Just a Transcript | **Integrated Slide OCR & Keyframes** |
    | **Trust Factor** | High Hallucination Risk | **Grounded (Fact-Checked via Timestamps)** |

    ---
    ### 🚀 The LectaLens Competitive Edge:
    * **👁️ Visual Awareness:** We use **Z-Score Statistical Profiling** to detect exactly when a slide changes. If a professor says *"Look at this graph,"* we actually look at it.
    * **🧩 Intelligence Partitioning:** We don't cut text randomly. We find the **Mathematical Elbow** in the conversation to create organic, logical chapters.
    * **🛡️ The Hallucination Firewall:** Every claim in the summary is anchored to a **[Timestamp]**. If the AI can't prove it happened at a specific second, it doesn't say it.
    * **📊 Dynamic Profiling:** We analyze the video's 'pulse' in the first 5% to adapt to its unique lighting and pace before processing.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.File(label="Upload Lecture (MP4)")
            run_btn = gr.Button("🚀 Execute", variant="primary")
            vitals_output = gr.Dataframe(label="Lecture Vitals", interactive=False)
            eval_output = gr.Dataframe(label="Intelligence Evaluation", interactive=False)

        with gr.Column(scale=2):
            log_output = gr.Textbox(label="Process Logs & AI Synthesis", lines=40)

    run_btn.click(fn=process_video_streaming, inputs=video_input, outputs=[log_output, vitals_output, eval_output])

demo.launch(debug=True)

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 94.0MiB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1313/796529141.py:219: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"), title="LectaLens") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6fa5b0ab1d535f1456.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



        Extract the knowledge from this lecture context.
        Structure your response EXACTLY like this:

        ### Summary: [Creative Title Based on Context]
        [Detailed 3-paragraph summary. Use **bolding** for key terms.
        IMPORTANT: You MUST cite the source time for every claim using [0.00s] format.]

        ---
        ### Main Thesis of the Lecture
        [One sentence bolded summary of the core message]

        ---
        ### Key Technical Takeaways
        * **[Concept Name]**: [Explanation] [Timestamp]
        * [Add more bullet points based on the richness of the context]

        CONTEXT:
        [14.16s]:  When we are hit with an unethical request, we usually only have a split second to react. (know you ve but we could spend more time together.   = you're but we could spend more time together.  Rather than just hit tongue-tied and shocked, we need to be ready to respond. have a few drinks and then well see 7  It seems too easy to just say no.  While it 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3552.81ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 10893.97ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12889.24ms
